# Twitter Scraper 

### Luiz Verheyen 

In [136]:
# imports

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import re
import time
import pandas as pd
from datetime import datetime
import os
import pyautogui
import time
import random
from dotenv import load_dotenv

In [137]:
# zorgen dat ik de env credentials kan gebruiken
load_dotenv() 

True

## Credentials needed for login
- email
- username
- password

In [138]:
my_twitter_email = os.getenv("my_twitter_email")
my_twitter_username = os.getenv("my_twitter_username")
my_twitter_password = os.getenv("my_twitter_password")

In [139]:
my_twitter_email = [acc.split(":") for acc in os.getenv("my_twitter_email").split(",")]
my_twitter_username = [acc.split(":") for acc in os.getenv("my_twitter_username").split(",")]
acc_index = 2 # default account

In [140]:
print(my_twitter_email)
print(my_twitter_username)

[['Luiz.verheyen@icloud.com'], ['0472863379']]
[['verheyen_l89639'], ['scraper0101'], ['scraper0102'], ['scraper0103'], ['scraper0104'], ['scraper0105']]


### options for the scraping configuration

In [141]:
options = uc.ChromeOptions() #initializing
options.add_argument("--start-maximized") # start screen volledig
options.add_argument("--disable-notifications") # disable alle notificaties die zouden binnenkomen

In [142]:
driver = uc.Chrome(version_main=145, options=options)

In [143]:
def login(acc_index, driver=driver):
    try:
        
        driver.get("https://twitter.com/login")
        time.sleep(2)  # wacht tot de pagina volledig geladen is
        
        email_input = driver.find_element(By.NAME, "text")
        email_input.send_keys(my_twitter_email[acc_index])
        time.sleep(1)
        email_input.send_keys(Keys.ENTER)
        time.sleep(3)
    except:
        print("Geen Username input gevonden of al ingevuld.")
        
    # 2 factor authentication
    try:
        username_input = driver.find_element(By.NAME, "text")
        username_input.send_keys(my_twitter_username[acc_index])
        time.sleep(1)
        username_input.send_keys(Keys.ENTER)
        time.sleep(3)
    except:
        print("no 2 factor authentication found or not needed.")
        
    # password in tikken
    try:
        password_input = driver.find_element(By.NAME, "password")
        password_input.send_keys(my_twitter_password)
        time.sleep(1)
        password_input.send_keys(Keys.ENTER)
        time.sleep(3)
    except:
        print("Geen wachtwoord invoer vereist of al ingelogd.")

In [144]:
login(acc_index=acc_index)

Geen Username input gevonden of al ingevuld.


In [145]:
def twitter_handle(since, until, username):
    driver.get(f"https://x.com/search?q=from:{username}%20since:{since}%20until:{until}%20-filter:replies&f=live")
    time.sleep(3)  # wachten tot pagina laadt
    # zorg dat cookies worden ge accepteert indien nodig:
    try:
        cookie_button = driver.find_element(By.XPATH, '//button[contains(., "Accept all cookies")]')
        cookie_button.click()
        print("Cookies geaccepteerd.")
        # time.sleep(2)
    except:
        print("Geen cookie-wall gevonden of al geaccepteerd.")

In [146]:
def human_scroll(driver, total_scroll=3000, step=300, pause=1):
    scrolled = 0
    while scrolled < total_scroll:
        driver.execute_script(f"window.scrollBy(0, {step});")
        scrolled += step
        time.sleep(pause)

In [147]:
def save_to_csv(tweets_data):
    df = pd.DataFrame(tweets_data, columns=["Date", "Username", "Content", "Replies", "Reposts", "Likes", "Bookmarks", "Views"])
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M')
    df['Time'] = pd.to_datetime(df['Date']).dt.time
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    print(f"{len(df)} tweets opgeslagen!")
    return df

In [148]:
def human_click_captcha(x_coord, y_coord):
    """Beweegt de muis naar de captcha en klikt op een menselijke manier."""
    print(f"PyAutoGUI probeert de captcha te klikken op ({x_coord}, {y_coord})...")
    
    # Beweeg de muis niet in een rechte lijn (menselijker)
    pyautogui.moveTo(
        x_coord + random.randint(-10, 10), 
        y_coord + random.randint(-10, 10), 
        duration=random.uniform(0.5, 1.5)
    )
    time.sleep(random.uniform(0.1, 0.4))
    pyautogui.click()
    time.sleep(2) # Wacht even tot Cloudflare verwerkt is

In [149]:
# usernames_we_wanna_scrape = ["tim_cook", "elonmusk"]
usernames_we_wanna_scrape = ["amazon", "nvidia", "Tesla", "tim_cook", "WhiteHouse"]
tweets_data = []

In [150]:
from datetime import datetime
import time
import re
import os
import pandas as pd
from selenium.webdriver.common.by import By

# Instellingen
since = "2010-01-01"
scroll_pause = 2
MAX_RETRY = 3

for user in usernames_we_wanna_scrape:
    print(f"\n--- Start scraping {user} ---")

    file_path = f"../../raw/tweets/{user}/{user}_tweets.csv"

    # ✅ CSV veilig laden
    if os.path.exists(file_path):
        df_user = pd.read_csv(file_path)
    else:
        df_user = pd.DataFrame()

    # ✅ last_date fix
    if df_user.empty:
        last_date = None
    else:
        last_date = str(df_user["date"].iloc[-1])[:10]

    until = datetime.today().strftime("%Y-%m-%d") if last_date is None else last_date

    tweets_ids = set()
    tweets_data = []
    retry = 0
    stop_scraping = False

    twitter_handle(username=user, since=since, until=until)

    while not stop_scraping:

        refresh_needed = False
        tweets_before = len(tweets_data)

        articles = driver.find_elements(By.XPATH, '//article[@role="article" and @data-testid="tweet"]')

        print(f"gevonden articles: {len(articles)} | tweets: {len(tweets_data)}")

        for article in articles:
            try:
                time_elem = article.find_element(By.XPATH, './/time')
                date_str = time_elem.get_attribute("datetime")

                tweet_date = datetime.fromisoformat(
                    date_str.replace("Z", "+00:00")
                ).replace(tzinfo=None)

                tweet_link = article.find_element(By.XPATH, './/time/..').get_attribute('href')
                tweet_id = tweet_link

                if tweet_id in tweets_ids:
                    continue

                if f"/{user}/" not in tweet_link:
                    continue

                # 📌 pinned
                try:
                    social_context = article.find_element(By.CSS_SELECTOR, "div[data-testid='socialContext']")
                    is_pinned = "Pinned" in social_context.text
                except:
                    is_pinned = False

                # 🛑 stopconditie
                if tweet_date.strftime("%Y-%m-%d") < since:
                    if is_pinned:
                        continue
                    print("✅ User volledig gescraped!")
                    stop_scraping = True
                    break

                # 📝 tekst
                try:
                    text = article.find_element(By.XPATH, './/div[@data-testid="tweetText"]').text
                except:
                    text = ""

                # 📊 stats
                try:
                    stats_group = article.find_element(By.XPATH, './/div[@role="group"]')
                    label = stats_group.get_attribute("aria-label")
                except:
                    label = ""

                def parse_stat(pattern, text):
                    match = re.search(pattern, text.lower())
                    if match:
                        return int(re.sub(r'[^\d]', '', match.group(1)))
                    return 0

                replies = parse_stat(r'(\d[\d\.,]*)\s+replies', label)
                reposts = parse_stat(r'(\d[\d\.,]*)\s+reposts', label)
                likes = parse_stat(r'(\d[\d\.,]*)\s+likes', label)
                bookmarks = parse_stat(r'(\d[\d\.,]*)\s+bookmarks', label)
                views = parse_stat(r'(\d[\d\.,]*)\s+views', label)

                tweets_data.append([
                    tweet_date, user, text,
                    replies, reposts, likes, bookmarks, views
                ])

                tweets_ids.add(tweet_id)

                print(f"✔ {tweet_date} | likes: {likes}")

                # 🔁 batch refresh
                if len(tweets_ids) % 363 == 0:
                    until = tweet_date.strftime("%Y-%m-%d")
                    print(f"Batch → nieuwe until: {until}")
                    refresh_needed = True
                    break

            except Exception as e:
                print("Fout:", e)
                continue

        # 🔁 batch refresh
        if refresh_needed:
            twitter_handle(username=user, since=since, until=until)
            continue

        # 🔁 stagnatie check
        tweets_after = len(tweets_data)

        if tweets_after == tweets_before:
            # Check of we vastzitten op het Cloudflare scherm
            if "Verify you are human" in driver.page_source or "Checking your browser" in driver.page_source:
                print("Captcha gedetecteerd!")
                
                # Gebruik PyAutoGUI om te klikken
                human_click_captcha(x_coord=715, y_coord=708)
                
                # Geef de pagina extra tijd om te herladen na de klik
                time.sleep(5) 
                
                continue # Probeer de loop opnieuw nu de captcha (hopelijk) weg is

            retry += 1
            print(f"⚠ Geen nieuwe tweets (retry {retry})")

            if retry >= MAX_RETRY:
                print("🔄 Rate limit vermoed → switch account")

                driver.delete_all_cookies()
                driver.execute_script("window.localStorage.clear();")
                driver.execute_script("window.sessionStorage.clear();")

                acc_index +=1
                login(acc_index=acc_index)

                retry = 0
                twitter_handle(username=user, since=since, until=until)

            if tweets_data:
                last_tweet_date = tweets_data[-1][0]
                until = last_tweet_date.strftime("%Y-%m-%d")
                twitter_handle(username=user, since=since, until=until)
            continue

        else:
            retry = 0

        # 🖱 scroll
        human_scroll(driver, total_scroll=2400, step=250, pause=0.2)
        time.sleep(scroll_pause)

    # 💾 opslaan
    if tweets_data:
        df = pd.DataFrame(tweets_data, columns=[
            "date", "username", "text",
            "replies", "reposts", "likes", "bookmarks", "views"
        ])

        df["date"] = pd.to_datetime(df["date"], errors="coerce")

        os.makedirs(f"../../raw/tweets/{user}", exist_ok=True)

        df.to_csv(file_path, index=False, mode="a", header=not os.path.exists(file_path))

        print(f"✅ {len(df)} tweets opgeslagen voor {user}")
    else:
        print(f"❌ Geen tweets gevonden voor {user}")

driver.close()
print("🎉 Klaar!")


--- Start scraping amazon ---
Cookies geaccepteerd.
gevonden articles: 2 | tweets: 0
✔ 2017-05-30 20:00:04 | likes: 30
✔ 2017-05-28 20:00:01 | likes: 103
gevonden articles: 2 | tweets: 2
⚠ Geen nieuwe tweets (retry 1)
Geen cookie-wall gevonden of al geaccepteerd.
gevonden articles: 0 | tweets: 2
⚠ Geen nieuwe tweets (retry 2)
Geen cookie-wall gevonden of al geaccepteerd.
gevonden articles: 5 | tweets: 2
✔ 2017-05-27 19:01:09 | likes: 27
✔ 2017-05-27 16:00:07 | likes: 24
✔ 2017-05-27 00:01:08 | likes: 21
✔ 2017-05-26 19:00:03 | likes: 43
✔ 2017-05-25 22:00:03 | likes: 73
gevonden articles: 4 | tweets: 7
⚠ Geen nieuwe tweets (retry 1)
Geen cookie-wall gevonden of al geaccepteerd.
gevonden articles: 6 | tweets: 7
✔ 2017-05-24 22:05:08 | likes: 23
✔ 2017-05-24 02:00:02 | likes: 18
✔ 2017-05-23 23:45:02 | likes: 29
✔ 2017-05-23 22:30:02 | likes: 15
✔ 2017-05-23 21:12:02 | likes: 50
✔ 2017-05-22 20:00:12 | likes: 35
gevonden articles: 5 | tweets: 13
⚠ Geen nieuwe tweets (retry 1)
Geen cooki

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=146.0.7680.165)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x3b7dd3
	0x3b7e14
	0x1c1db0
	0x1a08d3
	0x2350fb
	0x24b0d9
	0x22e7d6
	0x200049
	0x200e04
	0x616924
	0x611bf7
	0x62f5a0
	0x3d0f58
	0x3d891d
	0x3c0648
	0x3c0812
	0x3aa21a
	0x772c5d49
	0x77c6d81b
	0x77c6d7a1


In [ ]:
import pyautogui
import time

print("Zet je muis op de checkbox van de captcha...")
time.sleep(3)
print(f"Jouw coördinaten zijn: {pyautogui.position()}")

Zet je muis op de checkbox van de captcha...
Jouw coördinaten zijn: Point(x=715, y=708)
